# Canonical re-run of the TRACE-GNN reported arms

**What this notebook is for.** It produces the single designated set of results for every
TRACE-GNN arm the paper reports, so that each number, each score file and each validation
curve comes from the same run. It calls `rerun_canonical.py` and streams its progress here.

**Why it exists.** Three problems in the published results, all closed by one session.

| Problem | How it arose | What this fixes it with |
|---|---|---|
| The headline run was never validated | Notebook 15 compared its re-run against notebook 12 with a `gap < 0.002` tolerance, got 0.0096, printed **DIVERGENT, do not use these scores until resolved**, and the scores were used anyway | One canonical run, with every downstream artefact taken from it |
| Figure 7(b) plots a different execution from Table 4 | Notebook 15 passed no `val_curve_path`, so no curves exist for the reported run | `val_curve_path` is set for every arm, and the run asserts the curves cover all 17 folds |
| Two runs cannot support a variance claim | Only two executions exist, and one of them is a copy | `--repeats N` gives an actual standard deviation |

**Every run cell below is idempotent.** Each one means *make this complete*, not *start a
new run*. Nothing exists yet, it starts; something is half done, it continues from the last
finished fold; already done, it returns immediately. So you can re-run this notebook top to
bottom after a crash and simply arrive at the finished state. Training that has already
happened is never repeated and never overwritten.

That matters because this is a long job on a large graph. The `depth3` arm expands three
causal hops, and on the last and largest fold that subgraph approaches the whole 500k-node
graph. It has run out of memory on a 47 GB card. Section 3 estimates the peak before you
commit, and Section 5 gives that arm its own cell so it can be sent to a second GPU.

## 1. Settings

`REPEATS` is the only judgement call. `0` gives clean provenance but no variance estimate,
which leaves Section 5.4 with nothing to compare architecture effects against. `3` is the
smallest number that yields a usable standard deviation, and without it the architecture
contrasts cannot be told apart from run-to-run noise.

`OUT` pins the output folder. Set it to an existing `results/rerun_*` folder to continue
that run; leave it `None` to begin a new one. Because the run cells are idempotent, naming
the folder is what lets you resume, so prefer a fixed name over a fresh timestamp.

`DEVICE` is per cell, not global: the light arms and `depth3` are launched separately below
so they can go to different GPUs.

`PREPARED` is found by searching upward from the repo for `prepared/sparkov_clean_v1.csv`,
since that folder sits outside the repository.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json, textwrap
import pandas as pd

# --- repo root -----------------------------------------------------------------
REPO = Path.cwd() if (Path.cwd()/'rerun_canonical.py').exists() else Path.cwd().parent
assert (REPO/'rerun_canonical.py').exists(), f'rerun_canonical.py not found from {Path.cwd()}'
SCRIPT = REPO/'rerun_canonical.py'

# --- prepared data -------------------------------------------------------------
# rerun_canonical.py finds this itself, but resolve it here too so the path is visible
# and a mistake surfaces now rather than inside the subprocess.
PREPARED = None                      # <-- set an explicit Path here to override the search
if PREPARED is None:
    for base in [REPO, *REPO.parents]:
        cand = base/'prepared'/'sparkov_clean_v1.csv'
        if cand.exists():
            PREPARED = cand.parent
            break
assert PREPARED is not None, (
    'sparkov_clean_v1.csv not found in any prepared/ folder above '
    f'{REPO}. Set PREPARED explicitly above.')
PREPARED = Path(PREPARED).resolve()

# --- what to run ---------------------------------------------------------------
# The folder this notebook builds. Name it to make the run cells resumable.
OUT = REPO/'results'/'rerun_20260826_114717_m5'      # None = start a new timestamped folder

LIGHT_ARMS = 'main,uid_only,maxagg'  # 2 hops, modest memory
HEAVY_ARM  = 'depth3'                # 3 hops, this is the one that runs out of memory
REPEATS    = 3                       # extra runs of main, for the run-to-run sd

# GPU 0 usually also drives the desktop, which costs a little VRAM and can spike.
# Give the heavy arm a card of its own if there is one.
DEVICE_LIGHT = 'cuda:0'
DEVICE_HEAVY = 'cuda:1'

print('repo    :', REPO.resolve())
print('prepared:', PREPARED, f"({(PREPARED/'sparkov_clean_v1.csv').stat().st_size/1e6:.0f} MB csv)")
print('out     :', OUT if OUT else '(new timestamped folder)')
print('arms    :', LIGHT_ARMS, '+', HEAVY_ARM, '  repeats:', REPEATS)

## 2. Runner

`stream` echoes each line as the script prints it, because `subprocess.run` buffers and a
run this long would otherwise show nothing for hours.

`ensure` is the idempotent wrapper every run cell uses. It always passes `--resume`, which
on an absent folder simply starts a fresh run, on a partial folder trains only the missing
folds, and on a complete folder trains nothing. It returns the output folder so later cells
can read from it.

In [ ]:
def stream(cmd):
    """Run cmd, echo each line as it appears, return (exit_code, full_log)."""
    print('$', ' '.join(str(c) for c in cmd), '\n' + '-'*78)
    p = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        print(line, end='')
        lines.append(line)
    p.wait()
    print('-'*78, f'exit {p.returncode}')
    return p.returncode, ''.join(lines)


def ensure(arms, device, repeats=0, out=None, extra=()):
    """Make `arms` complete in the output folder. Safe to re-run at any time.

    Always passes --resume, so: no folder -> fresh run; partial -> only the missing
    folds; complete -> returns without training. Never overwrites finished work.
    """
    global OUT
    out = out or OUT
    cmd = [sys.executable, SCRIPT, '--prepared', PREPARED, '--arms', arms,
           '--device', device, '--resume', *extra]
    if out:      cmd += ['--out', out]
    if repeats:  cmd += ['--repeats', str(repeats)]
    rc, log = stream(cmd)
    assert rc == 0, f'{arms} failed, read the ABORT or traceback above'
    if out is None:                      # first cell of a fresh run names the folder
        OUT = Path([l.split(':', 1)[1].strip() for l in log.splitlines()
                    if l.strip().startswith('output folder')][0])
        print('\nnew output folder:', OUT)
    return Path(out or OUT), log

## 3. Dry run and memory preflight

Loads the frame, builds the folds, builds the graph, locates the saved XGBoost baseline and
runs every assertion, then stops before training. It creates no output folder.

**What you should see.** 500,000 rows, 11 features, 17 folds, about 9.9 million edges at
roughly 19.8 per node, and 54 contemporaneous edges. If the feature count is not 11 or the
edge count is far from 9.9 M, stop and investigate rather than starting the real run.

It also samples one real training batch from the largest fold and projects peak VRAM per
arm. `depth3` needs roughly three times what the two-hop arms need, on the same fold. If
the projection exceeds free memory the run will very likely die on the last folds, so move
that arm to an emptier card before spending the hours rather than after.

In [ ]:
rc, log = stream([sys.executable, SCRIPT, '--prepared', PREPARED,
                  '--arms', f'{LIGHT_ARMS},{HEAVY_ARM}', '--device', DEVICE_LIGHT,
                  '--dry-run'])
assert rc == 0, 'dry run failed, read the ABORT line above before going further'

## 4. The light arms

`main`, `uid_only` and `maxagg`, all two-hop. Roughly 9 h, 1 h and 7 h.

Output goes to `OUT`. Nothing outside that folder is touched, and any arm already finished
there is skipped rather than repeated.

### What the script guards against, so you know what a failure means

| Assertion | The mistake it catches |
|---|---|
| every `TrainConfig` field | `TrainConfig()` defaults to `lr=1e-3`, `batch_size=8192`, `use_swa=True`. None is the reported configuration, and `use_swa=True` silently disables early stopping. A previous session lost its results exactly this way |
| `edge_dim == 5` | passing `('uid',)` instead of `('uid','__none__')` gives `edge_dim=4`, a different `edge_bias` layer and a different parameter count |
| 17 folds returned | a stale `foldckpt.csv` making the runner skip folds without being asked |
| append files agree | a half-written fold, which resuming would duplicate |
| 338,456 scored rows, 1,276 positives | a truncated or appended scores file |
| curves cover 17 folds | the Figure 7(b) problem recurring |
| no fold reaches the 200 epoch ceiling | early stopping not binding, the downstream symptom of `use_swa=True` |
| one writer per arm | two processes resuming the same arm and both appending |

In [ ]:
OUT, _ = ensure(LIGHT_ARMS, DEVICE_LIGHT)
print('\noutput folder:', OUT)

## 5. The deep arm

`depth3` on its own, because it is the expensive and fragile one: three hops instead of
two, about 4x the cost per epoch, and it is where the run died last time with
`torch.OutOfMemoryError` on week 24, the largest fold. Roughly 33 h.

It is separated for two reasons. It can be pointed at a different GPU, `DEVICE_HEAVY`
above, so it is not competing with the desktop or with the light arms. And if it does run
out of memory, only this cell fails: the other three arms are already safe on disk, and
re-running this cell alone picks up from the last finished fold rather than starting over.

Run it on its own, or at the same time as Section 4 if you have two cards. Locking is per
arm, so concurrent cells writing different arms into the same folder is safe.

In [ ]:
_ = ensure(HEAVY_ARM, DEVICE_HEAVY)

## 6. Repeats, the noise baseline

Three more executions of `main` at seeds 43, 44 and 45, so the standard deviation reflects
initialisation and batch order. Under a single fixed seed it would only measure GPU
nondeterminism, which is far smaller than the effects Section 5.4 compares against and
would overstate their significance. Roughly 27 h, or half that spread over two cards.

**Section 5.4 needs this.** The architecture contrasts are small, and without an estimate
of run-to-run variation there is no way to say whether they are real. Skip this and the
comparison has no denominator.

Finishing this cell also rebuilds `derived/` and `COMPARISON.md` across all four arms.

In [ ]:
_ = ensure('main', DEVICE_LIGHT, repeats=REPEATS)

## 7. Read the comparison

`COMPARISON.md` is the file that decides what happens to the manuscript. It tabulates the
paper's value against this run's value for every arm and every derived quantity, then
states plainly whether anything moved.

- **If nothing moved by 0.001**, only provenance changed. No manuscript number needs editing.
- **If an arm moved**, it lists exactly which tables, figures and sections go stale.
  Section 5.3's protocol study is a separate set of executions and is not affected either way.

In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUT/'COMPARISON.md').read_text(encoding='utf8')))

## 8. Numbers at a glance

`hours` accumulates across segments, so an arm that was resumed reports its true total
rather than the time of its final fold. `segments > 1` means that arm was not produced in
one sitting; `run_manifest.json` records what each segment trained.

In [ ]:
s = pd.read_csv(OUT/'canonical_summary.csv')
cols = ['arm','mean_ap','paper_ap','delta_vs_paper','epochs_min','epochs_median',
        'epochs_max','hours'] + [c for c in ('segments','resumed') if c in s.columns]
display(s[cols].round(4))

pt = OUT/'derived'/'paired_tests.csv'
if pt.exists():
    print('\npaired tests'); display(pd.read_csv(pt))

rs = OUT/'repeat_summary.csv'
if rs.exists():
    r = pd.read_csv(rs)
    print(f"\nrun-to-run over {len(r)} executions of the main arm")
    print(f"  mean {r.mean_ap.mean():.4f}   sd {r.mean_ap.std(ddof=1):.4f}   "
          f"range {r.mean_ap.min():.4f} to {r.mean_ap.max():.4f}")
    print('  quote the sd. This replaces the two-run 0.010 figure.')
else:
    print('\nno repeats, so run-to-run variation is NOT estimated.')
    print('Run section 6 before interpreting any architecture contrast.')

seg = json.loads((OUT/'run_manifest.json').read_text())
if len(seg.get('execution_segments', [])) > 1:
    print(f"\nproduced over {len(seg['execution_segments'])} execution segments:")
    for e in seg['execution_segments']:
        print(f"  {e['utc']}  {e['mode']:6}  arms={e.get('arms')}")
    print(' ', textwrap.fill(seg.get('provenance_note',''), 96, subsequent_indent='  '))

## 9. Redraw Figures 7 and 8 from this run

`fig_5_4.py` now takes `--canonical`, which reads every arm's `per_fold.csv` and the
validation curves from this folder. Nothing needs copying into `results/convergence/`, and
Table 4, Figure 7(b) and Figure 8 all come from the same set of results, which closes the
provenance gap.

Three things changed in that script:

- **No hard-coded arm means.** It used to assert each arm against a number copied from the
  manuscript and abort with *"Wrong file"* the moment a run was repeated. It now
  cross-checks each `per_fold.csv` against `canonical_summary.csv`, which catches a
  truncated or foreign file without encoding any particular result.
- **The median-stop line is computed from the data**, not fixed at 76 epochs. This run
  stops at 83.
- **The Figure 8 noise band is measured, not assumed.** Each row is a difference between
  two single executions, so the band is drawn at `sqrt(2) x SD` using the standard
  deviation from `repeat_summary.csv`. Any effect inside it is one that repeated runs of
  the same configuration could have produced on their own.

This cell overwrites `figures/F7_folds_convergence.*` and `figures/F8_effect_sizes.*`.
The old behaviour still works if you drop `--canonical`, so the previous figures remain
reproducible.

In [ ]:
fig_script = REPO/'figures'/'make'/'fig_5_4.py'
FIGDIR = REPO/'figures'          # writes F7_folds_convergence and F8_effect_sizes
rc, log = stream([sys.executable, fig_script, str(REPO/'results'), str(FIGDIR),
                  '--canonical', str(OUT)])
assert rc == 0, 'figure script failed, see above'

from IPython.display import Image, display
for f in ('F7_folds_convergence.png', 'F8_effect_sizes.png'):
    print(f'\n{f}')
    display(Image(filename=str(FIGDIR/f)))

## 10. Notes

Record here what you observed, anything that looked wrong, and what you decided. Future you
will want to know why this run exists and whether its numbers went into the paper.

- date run:
- machine and GPU:
- canonical main AP:
- did anything move by more than 0.001:
- run-to-run sd, and which contrasts survive it:
- manuscript updated (yes/no, which sections):
- anything unexpected: